# Advanced Feature: Collaborative Storytelling for Long Narratives

This notebook demonstrates how TinyTroupe's `TinyStory` class can be used to generate and collaboratively guide a relatively long narrative involving multiple agents. The `TinyStory` facilitator helps maintain coherence and allows the user to steer the plot at key moments.

The scenario involves:
1. Creating a diverse cast of characters: a few financial professionals and an alien named Zog who has just arrived on Earth.
2. Setting an initial scene where these characters meet unexpectedly.
3. Using the `TinyStory` object, initialized with the simulation `TinyWorld`, to:
    a. Start the story based on the initial premise.
    b. Periodically prompt the `TinyStory` object to generate continuations, guiding the narrative with high-level instructions (e.g., "something surprising happens," "introduce a new challenge," "reach a resolution").
4. Running the simulation in steps after each story continuation to allow agents to react and interact based on the evolving narrative.

This example showcases how TinyTroupe can be a tool for creative writing, plot generation, or simulating complex, evolving scenarios where narrative control is desired.

## 1. Setup and Imports

We import necessary classes: `TinyPerson` for agents, `TinyPersonFactory` for creating them, `TinyWorld` for their environment, and crucially, `TinyStory` for narrative control. Other utilities are also imported.

In [ ]:
import json
import sys
import csv
# If running from 'examples/advanced_features/', this adds the project root to Python path.
sys.path.insert(0, '../..') 

import tinytroupe # Initializes configuration, logging, etc.
from tinytroupe.openai_utils import force_api_type # Utility if needing to switch API type (e.g. to Azure)
from tinytroupe.factory import TinyPersonFactory
from tinytroupe.agent import TinyPerson # TinyToolUse not used in this example
from tinytroupe.environment import TinyWorld
# control module not directly used in this example notebook
# from tinytroupe import control 
# ResultsExtractor, ResultsReducer, TinyEnricher, ArtifactExporter, TinyWordProcessor not used here
# from tinytroupe.extraction import ResultsExtractor, ResultsReducer
# from tinytroupe.enrichment import TinyEnricher
# from tinytroupe.extraction import ArtifactExporter
# from tinytroupe.tools import TinyWordProcessor
from tinytroupe.steering import TinyStory
import tinytroupe.utils as utils

# Example agent creation helpers (though we mostly use factory here)
from tinytroupe.examples import create_lisa_the_data_scientist, create_oscar_the_architect, create_marcos_the_physician

## 2. Create the Main Characters

We'll use `TinyPersonFactory` to create a few financial professionals and an alien character, Zog. The factory context describes a financial services firm, which will influence the personas of the human characters.

In [ ]:
factory_context =\
"""
InvesTastic is a financial services firm that specializes in providing highly customized investment advice for
discerning clients. The firm has a reputation for digging deep into the financials of companies and industries,
and for providing clients with a comprehensive understanding of the risks and rewards of various investment
opportunities. Clients say that InvesTastic is the company which best tailors its advice to their individual
needs and interests.
"""
factory = TinyPersonFactory(factory_context)

print("Generating financial professionals...")
person_1 = factory.generate_person("A financial analyst specialized in commodities.")
print(f"- {person_1.name}: {utils.wrap_text(person_1.minibio())}")

person_2 = factory.generate_person("A financial advisor, who interacts directly with customers to help with their investment needs.")
print(f"- {person_2.name}: {utils.wrap_text(person_2.minibio())}")

person_3 = factory.generate_person("A customer of InvesTastic, who is looking for new investment opportunities.")
print(f"- {person_3.name}: {utils.wrap_text(person_3.minibio())}")

# Generate Zog, the alien
print("\nGenerating Zog the alien...")
zog = factory.generate_person("Zog, an alien from a technologically advanced civilization. Zog is curious, friendly, and has just arrived on Earth with the mission to understand human cultures, especially their economic and social structures. Zog is unfamiliar with Earth customs but is a quick learner.")
if zog:
    print(f"- {zog.name}: {utils.wrap_text(zog.minibio())}")
else:
    print("Failed to generate Zog.")

## 3. Set up the World and Story Facilitator

Create a `TinyWorld` and add the agents. Then, initialize the `TinyStory` object which will help us guide the narrative.

In [ ]:
world = TinyWorld("Rio de Janeiro Beach", [person_1, person_2, person_3, zog] if zog else [person_1, person_2, person_3])
world.make_everyone_accessible()
print(f"World '{world.name}' created with agents: {[agent.name for agent in world.agents]}")

# Define the beginning of our story
story_beginning =\
          """
            You ({}, {}, and {}) were attending a finance conference in the beautiful city of Rio de Janeiro, Brazil. 
            While taking a break and walking along Copacabana beach, the most unexpected event of your lives occurred: 
            a small, sleek alien spaceship silently descended and landed right in front of you on the sand. 
            A hatch hissed open, and a friendly-looking alien ({}) stepped out. 
            The alien introduced itself as Zog, explaining it was on a peaceful mission to learn more about Earth's cultures, 
            particularly its economic systems and how humans interact around value and resources. 
            You were all, understandably, astonished but also intrigued by this incredible encounter and cautiously decided to engage with Zog.
          """.format(person_1.name, person_2.name, person_3.name, zog.name if zog else "the alien")

story = TinyStory(world)
print("TinyStory facilitator initialized.")

### 3.1. Helper Function for Story Continuation

This function will make it easier to prompt `TinyStory` for a narrative update, broadcast it to the agents, and run the simulation for a couple of steps to see their reactions.

In [ ]:
def continue_story_and_run(continuation_requirements="Continue the story in an interesting and logical way, ensuring all characters have a chance to be involved.", run_steps=2):
    """Prompts TinyStory to continue the narrative, broadcasts it, and runs the world."""
    print("\n--- Requesting Story Continuation ---")
    print(f"Prompt for continuation: {continuation_requirements}")
    continuation = story.continue_story(continuation_requirements)
    
    if continuation:
        print("\n--- Story Continuation Received ---")
        print(utils.wrap_text(continuation))
        # Broadcast the story continuation to all agents in the world
        world.broadcast(continuation)   
        # Run the world for a few steps to see agent reactions
        world.run(run_steps)
    else:
        print("Failed to generate story continuation.")

## 4. Let's Tell the Story!

Start by broadcasting the initial story premise to the agents and run the simulation to get their initial reactions.

In [ ]:
print("--- Starting the Story: Zog Meets Earth Investors ---")
print("\nInitial Story Premise:")
print(utils.wrap_text(story_beginning))
world.broadcast(story_beginning)
world.run(2) # Let agents react to the initial situation

Now, let's use our helper function to guide the story.

In [ ]:
continue_story_and_run("Zog is very curious about the concept of 'investment' and asks the financial professionals to explain it. One of them takes the lead.", run_steps=2)

In [ ]:
continue_story_and_run("Suddenly, a news alert pops up on Evelyn's phone about a massive, unexpected global market crash, creating panic and disbelief among the humans.", run_steps=3)

In [ ]:
continue_story_and_run("Zog, using its advanced technology, offers a unique perspective or a piece of information that could explain the market crash or offer a surprising solution, leaving the human investors astounded.", run_steps=3)

In [ ]:
continue_story_and_run("The story should reach a satisfying conclusion. The group decides on a collaborative next step based on Zog's revelation, and they reflect on the encounter and what they've learned about finance, ethics, and perhaps even intergalactic cooperation.", run_steps=2)

## 5. Review the Full Narrative

You can review the entire sequence of interactions printed above to see the full story that unfolded. Each agent's `pp_current_interactions()` could also be called for their individual perspectives.